# Customer Churn Intelligence — Focused EDA

## Objective

Explore relationships between features and churn using the **cleaned** dataset. This step builds on initial inspection (`01_eda`), cleaning (`02`), and feature/target separation (`03`).

**Stage:** Step 6 — Focused EDA (no train/test split, encoding, or modeling).

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_ROOT = Path("..").resolve()
FIGURES_DIR = PROJECT_ROOT / "reports" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_cleaning import TARGET_COL
from src.data_separation import (
    CATEGORICAL_FEATURE_COLS,
    NUMERIC_FEATURE_COLS,
    load_cleaned_data,
    separate_features_target_id,
)

plt.style.use("seaborn-v0_8-whitegrid")
CHURN_PALETTE = {"No": "#4C72B0", "Yes": "#DD8452"}

In [ ]:
df = load_cleaned_data()
separated = separate_features_target_id(df)

eda_df = separated.X.copy()
eda_df[TARGET_COL] = separated.y.values

print(f"Rows: {len(eda_df):,}")
print(f"Features: {eda_df.shape[1] - 1}")

## 1. Target Distribution

In [ ]:
churn_counts = eda_df[TARGET_COL].value_counts()
churn_pct = (eda_df[TARGET_COL].value_counts(normalize=True) * 100).round(2)

target_summary = pd.DataFrame({"Count": churn_counts, "Percent": churn_pct})
display(target_summary)

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(churn_counts.index, churn_counts.values, color=[CHURN_PALETTE[k] for k in churn_counts.index])
ax.set_title("Churn Class Distribution")
ax.set_xlabel("Churn")
ax.set_ylabel("Customer Count")
for bar, pct in zip(bars, churn_pct.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 50, f"{pct:.1f}%", ha="center")
plt.tight_layout()
fig.savefig(FIGURES_DIR / "01_churn_distribution.png", dpi=120)
plt.show()

## 2. Numeric Features vs Churn

In [ ]:
numeric_summary = (
    eda_df.groupby(TARGET_COL)[NUMERIC_FEATURE_COLS]
    .agg(["mean", "median", "std"])
    .round(2)
)
numeric_summary

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 9))
axes = axes.ravel()

for ax, col in zip(axes, NUMERIC_FEATURE_COLS):
    for label in ["No", "Yes"]:
        subset = eda_df.loc[eda_df[TARGET_COL] == label, col]
        ax.hist(subset, bins=30, alpha=0.55, label=label, color=CHURN_PALETTE[label])
    ax.set_title(f"{col} by Churn")
    ax.set_xlabel(col)
    ax.set_ylabel("Count")
    ax.legend()

plt.suptitle("Numeric Feature Distributions by Churn", y=1.02)
plt.tight_layout()
fig.savefig(FIGURES_DIR / "02_numeric_distributions_by_churn.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 9))
axes = axes.ravel()

for ax, col in zip(axes, NUMERIC_FEATURE_COLS):
    eda_df.boxplot(column=col, by=TARGET_COL, ax=ax, patch_artist=True)
    ax.set_title(col)
    ax.set_xlabel("Churn")
    ax.set_ylabel(col)

plt.suptitle("Numeric Feature Boxplots by Churn", y=1.02)
plt.tight_layout()
fig.savefig(FIGURES_DIR / "03_numeric_boxplots_by_churn.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
corr = eda_df[NUMERIC_FEATURE_COLS].corr()
display(corr.round(3))

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(NUMERIC_FEATURE_COLS)), NUMERIC_FEATURE_COLS, rotation=45, ha="right")
ax.set_yticks(range(len(NUMERIC_FEATURE_COLS)), NUMERIC_FEATURE_COLS)
for i in range(len(NUMERIC_FEATURE_COLS)):
    for j in range(len(NUMERIC_FEATURE_COLS)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", color="black", fontsize=9)
ax.set_title("Numeric Feature Correlation Matrix")
fig.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
fig.savefig(FIGURES_DIR / "04_numeric_correlation.png", dpi=120)
plt.show()

## 3. Categorical Features — Churn Rate by Category

In [ ]:
def churn_rate_table(df: pd.DataFrame, column: str) -> pd.DataFrame:
    """Compute customer count and churn rate (%) for each category level."""
    grouped = df.groupby(column, observed=True)
    out = pd.DataFrame({
        "Customers": grouped.size(),
        "Churn Rate (%)": (grouped[TARGET_COL].apply(lambda s: (s == "Yes").mean()) * 100).round(2),
    }).sort_values("Churn Rate (%)", ascending=False)
    return out


KEY_CATEGORICAL = [
    "Contract",
    "InternetService",
    "PaymentMethod",
    "PaperlessBilling",
    "TechSupport",
    "OnlineSecurity",
]

for col in KEY_CATEGORICAL:
    print(f"\n--- {col} ---")
    display(churn_rate_table(eda_df, col))

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.ravel()

for ax, col in zip(axes, KEY_CATEGORICAL):
    rates = churn_rate_table(eda_df, col)["Churn Rate (%)"]
    rates.plot(kind="barh", ax=ax, color="#4C72B0")
    ax.set_title(f"Churn Rate by {col}")
    ax.set_xlabel("Churn Rate (%)")
    ax.invert_yaxis()

plt.suptitle("Churn Rate by Key Categorical Features", y=1.02)
plt.tight_layout()
fig.savefig(FIGURES_DIR / "05_churn_rate_key_categoricals.png", dpi=120, bbox_inches="tight")
plt.show()

## 4. Tenure Bands vs Churn

In [ ]:
eda_df["tenure_band"] = pd.cut(
    eda_df["tenure"],
    bins=[-1, 12, 24, 48, 72],
    labels=["0-12 mo", "13-24 mo", "25-48 mo", "49-72 mo"],
)

tenure_churn = churn_rate_table(eda_df, "tenure_band")
display(tenure_churn)

fig, ax = plt.subplots(figsize=(7, 4))
tenure_churn["Churn Rate (%)"].plot(kind="bar", ax=ax, color="#DD8452")
ax.set_title("Churn Rate by Tenure Band")
ax.set_xlabel("Tenure Band")
ax.set_ylabel("Churn Rate (%)")
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
plt.tight_layout()
fig.savefig(FIGURES_DIR / "06_churn_rate_by_tenure_band.png", dpi=120)
plt.show()

## 5. All Categorical Features — Churn Rate Overview

In [ ]:
overview_rows = []
for col in CATEGORICAL_FEATURE_COLS:
    table = churn_rate_table(eda_df, col).reset_index()
    table["Feature"] = col
    table = table.rename(columns={col: "Category"})
    overview_rows.append(table)

churn_overview = pd.concat(overview_rows, ignore_index=True)
churn_overview = churn_overview.sort_values("Churn Rate (%)", ascending=False)
display(churn_overview.head(15))
display(churn_overview.tail(5))

## Focused EDA Findings

Summary based only on the cleaned dataset (7,043 customers):

### Target
- **Class imbalance:** 73.5% No churn vs 26.5% Yes churn — PR-AUC will be important alongside ROC-AUC.

### Numeric features
- **tenure:** Churned customers have lower average tenure (No: ~37.6 mo vs Yes: ~18.0 mo). Churn rate drops sharply as tenure increases (0–12 mo: ~47.4% → 49–72 mo: ~9.5%).
- **MonthlyCharges:** Churned customers pay higher monthly charges on average (No: ~61.3 vs Yes: ~74.4).
- **TotalCharges:** Non-churned customers have higher average total charges (No: ~2,550 vs Yes: ~1,532), consistent with longer tenure.
- **SeniorCitizen:** Higher churn among seniors (0: ~23.6% vs 1: ~41.7%).
- **Correlation:** Strong `tenure` ↔ `TotalCharges` (0.83); moderate `MonthlyCharges` ↔ `TotalCharges` (0.65). Multicollinearity may affect linear models.

### Categorical features (highest churn-rate signals)
- **Contract:** Month-to-month ~42.7% vs two-year ~2.8%.
- **InternetService:** Fiber optic ~41.9% vs DSL ~19.0% vs no internet ~7.4%.
- **PaymentMethod:** Electronic check ~45.3% vs automatic card/transfer ~15–17%.
- **PaperlessBilling:** Yes ~33.6% vs No ~16.3%.
- **TechSupport / OnlineSecurity:** "No" ~42% vs "Yes" ~15% ("No internet service" ~7.4%).

### Implications for next steps (not performed here)
- Class imbalance handling will likely be needed.
- Contract, tenure, internet service, and payment method appear strongly associated with churn.
- Sentinel categories require careful encoding during preprocessing.
- **No train/test split or modeling performed in this notebook.**